In [1]:
!pip install langchain_community pypdf langchain
!pip install langchain_huggingface faiss-cpu langchain_groq sentence-transformers

  Using cached langchain_community-0.3.30-py3-none-any.whl.metadata (3.0 kB)
  Using cached pypdf-6.1.0-py3-none-any.whl.metadata (7.1 kB)
  Using cached langchain-0.3.27-py3-none-any.whl.metadata (7.8 kB)
  Using cached langchain_core-0.3.76-py3-none-any.whl.metadata (3.7 kB)
  Using cached sqlalchemy-2.0.43-cp313-cp313-win_amd64.whl.metadata (9.8 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached pyyaml-6.0.3-cp313-cp313-win_amd64.whl.metadata (2.4 kB)
  Using cached aiohttp-3.12.15-cp313-cp313-win_amd64.whl.metadata (7.9 kB)
  Using cached tenacity-9.1.2-py3-none-any.whl.metadata (1.2 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl.metadata (25 kB)
  Using cached pydantic_settings-2.11.0-py3-none-any.whl.metadata (3.4 kB)
  Using cached langsmith-0.4.31-py3-none-any.whl.metadata (14 kB)
  Using cached httpx_sse-0.4.1-py3-none-any.whl.metadata (9.4 kB)
  Using cached numpy-2.3.3-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached lan

In [3]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('C:/Users/Rakesh/Desktop/pr/pr/data/AI Training Document.pdf')

documents = loader.load()
print(f"Loaded { len(documents)} documents")

Loaded 20 documents


In [4]:
import re

def preprocess_text(text):
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text)
    # Remove page numbers
    text = re.sub(r'Page \d+', '', text)
    # Remove special characters
    text = re.sub(r'[^\w\s\.\,\!\?]', '', text)
    return text.strip()

# Apply to documents
for doc in documents:
    doc.page_content = preprocess_text(doc.page_content)


In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)
docs = text_splitter.split_documents(documents)
print(f"split into {len(docs)} chunks")

split into 273 chunks


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"device": "cpu"}
)

c:\Users\Rakesh\Desktop\pr\pr\.venv_\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Rakesh\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling ba

In [8]:
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(docs,embeddings)
print("Vector store created")


Vector store created


In [9]:
from re import search


retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k":5}
)

query = "tell me about ebay"
retrieved_docs = retriever.get_relevant_documents(query)
print(f"Retrieved {len(retrieved_docs)} documents")

C:\Users\Rakesh\AppData\Local\Temp\ipykernel_18320\826398262.py:10: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  retrieved_docs = retriever.get_relevant_documents(query)


Retrieved 5 documents


In [ ]:
from langchain_groq import ChatGroq
from langchain.chains import RetrievalQA


# Initialize LLM
llm = ChatGroq(
    temperature=0,                 # Deterministic responses
    model_name="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY
)

# Create RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",            # Combine all context in one prompt
    retriever=retriever
)

# Generate response
response = rag_chain.invoke({"query": query})
print(response["result"])


**eBay – a quick overview**

| Topic | What eBay does |
|-------|----------------|
| **Core services** | eBay offers a marketplace where individuals and businesses can list items for sale. The platform provides tools for pricing, shipping, listing creation, and sourcing. eBay also uses artificial‑intelligence (AI) powered tools to help users customize and personalize their experience. |
| **Vehicle sales** | eBay is **not** a vehicle broker, dealer, or agent. It does not maintain an inventory of vehicles, nor does it buy, sell, or negotiate vehicle sales on behalf of buyers or sellers. |
| **Legal entities** | Depending on where you live, eBay’s services are provided by different entities:<br>• **Canada** – eBay Singapore Services Private Limited, 1 Raffles Quay, 18 00, Singapore 048583.<br>• **India** – eBay Marketplaces GmbH, Helvetiastrasse 1517, CH3005, Bern, Switzerland.<br>• **Other countries** – the applicable eBay entity is specified in the User Agreement. |
| **International s

In [ ]:
# from langchain.evaluation.qa import QAEvalChain

# # Prepare evaluation data
# examples = [{"query": query, "answer": "Ground truth answer"}]
# predictions = [{"query": query, "result": response["result"]}]

# # Evaluate
# eval_chain = QAEvalChain.from_llm(llm)
# graded_outputs = eval_chain.evaluate(examples, predictions)
# print(graded_outputs)


In [ ]:
python -m streamlit run app.py